<div style="background:linear-gradient(135deg,#4a044e 0%,#a21caf 55%,#e879f9 100%);border-radius:18px;padding:34px 30px;color:#fff;font-family:Inter,Segoe UI,sans-serif">
  <div style="font-size:12px;letter-spacing:3px;color:#f5d0fe;font-weight:700;text-transform:uppercase">Chapter 149 &middot; Advanced &amp; Applied Topics</div>
  <div style="font-size:34px;font-weight:900;line-height:1.1;margin:10px 0 6px">Big Data &amp; Scaling &middot; Challenge Solutions</div>
  <div style="font-size:15px;color:#fae8ff;max-width:740px;line-height:1.6">Worked solutions: shrink the memory footprint, aggregate a file in chunks, compare CSV with Parquet, write MapReduce by hand, and diagnose then repair data skew.</div>
</div>

In [ ]:
import numpy as np, pandas as pd, os, time, tempfile
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi":110,"axes.grid":True,"grid.alpha":0.25,"font.size":11})
FU, BL, GR, RD, AM = "#a21caf", "#2563eb", "#16a34a", "#dc2626", "#d97706"
BASE = "https://raw.githubusercontent.com/johnfisher-ai/Statistics-Data-Science-AI-Visual-Book/main/data/"
fn = "big-data-and-scaling--ride-logs.xlsx"
try:
    df = pd.read_excel("../../data/" + fn)
except FileNotFoundError:
    df = pd.read_excel(BASE + fn)
print(df.shape)
print(df.head(3).to_string())

# A scratch folder for the larger files we build below. Nothing here is written to the book repo.
TMP = tempfile.gettempdir()
print("\nscratch folder:", TMP)

## Challenge 1 &middot; Cut the memory footprint in half
Downcast the types and report the saving, then confirm the numbers did not move.

In [ ]:
opt = df.copy()
for col in opt.select_dtypes("object"):
    if opt[col].nunique() / len(opt) < 0.5: opt[col] = opt[col].astype("category")
for col in opt.select_dtypes("integer"): opt[col] = pd.to_numeric(opt[col], downcast="integer")
for col in opt.select_dtypes("float"):   opt[col] = pd.to_numeric(opt[col], downcast="float")
b, a = df.memory_usage(deep=True).sum(), opt.memory_usage(deep=True).sum()
print(f"{b/1e6:.2f} MB -> {a/1e6:.2f} MB   ({100*(1-a/b):.0f}% saved)")
print("mean fare unchanged:", round(float(df.fare.mean()), 4), "vs", round(float(opt.fare.mean()), 4))

## Challenge 2 &middot; Average trip distance by vehicle type, in chunks
Never hold the whole file. Remember that means carrying sums and counts, not means.

In [ ]:
big = pd.concat([df]*20, ignore_index=True)
p = os.path.join(TMP, "sol_rides.csv"); big.to_csv(p, index=False)
partials = [ch.groupby("vehicle_type").distance_km.agg(["sum", "count"]) for ch in pd.read_csv(p, chunksize=50_000)]
out = pd.concat(partials).groupby(level=0).sum()
out["avg_km"] = out["sum"] / out["count"]
print(out[["count", "avg_km"]].round(2).to_string())
print("\nmatches the full-file answer?",
      np.allclose(out.avg_km.sort_index(), big.groupby("vehicle_type").distance_km.mean().sort_index()))

## Challenge 3 &middot; How much does Parquet actually buy you?
Compare size and the time to read a single column.

In [ ]:
q = os.path.join(TMP, "sol_rides.parquet"); big.to_parquet(q, index=False)
cs, ps = os.path.getsize(p)/1e6, os.path.getsize(q)/1e6
t0 = time.perf_counter(); pd.read_csv(p, usecols=["fare"]); tc = time.perf_counter()-t0
t0 = time.perf_counter(); pd.read_parquet(q, columns=["fare"]); tp = time.perf_counter()-t0
print(f"size    CSV {cs:.1f} MB vs Parquet {ps:.1f} MB  ({cs/ps:.1f}x)")
print(f"one col CSV {tc:.3f}s   vs Parquet {tp:.3f}s     ({tc/tp:.0f}x)")
print("\nThe CSV must still scan every byte to find one column. Parquet just seeks to it.")

## Challenge 4 &middot; MapReduce for the busiest hour
Same three steps, a different key: count rides per pickup hour, then name the peak.

In [ ]:
from collections import defaultdict
pairs = [(r.pickup_hour, 1) for r in df.itertuples()]      # MAP
sh = defaultdict(list)
for k, v in pairs: sh[k].append(v)                          # SHUFFLE
counts = pd.Series({k: sum(v) for k, v in sh.items()}).sort_index()   # REDUCE
print(counts.to_string())
print(f"\nbusiest hour: {counts.idxmax()}:00 with {counts.max():,} rides")
print("matches value_counts?", counts.equals(df.pickup_hour.value_counts().sort_index()))
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(counts.index, counts.values, color=FU); ax.set_xlabel("pickup hour"); ax.set_ylabel("rides")
ax.set_title("Two rush hours, found by MapReduce"); plt.tight_layout(); plt.show()

## Challenge 5 &middot; Find the skew, then fix it
Report how lopsided the partitions are, salt the hot key, and confirm the answer survives.

In [ ]:
parts = df.city.value_counts()
imbalance = parts.max() / parts.mean()
print(f"largest partition {parts.max():,} rows, average {parts.mean():,.0f}  ->  imbalance {imbalance:.1f}x")
for SALT in [1, 2, 4, 8]:
    salted = df.city + "#" + np.random.default_rng(0).integers(0, SALT, len(df)).astype(str)
    v = salted.value_counts()
    print(f"  salt={SALT}: {len(v):>2} partitions, largest holds {100*v.max()/len(df):4.1f}% of rows")
salted = df.city + "#" + np.random.default_rng(0).integers(0, 8, len(df)).astype(str)
st1 = df.groupby(salted).fare.sum()
final = st1.groupby([k.split("#")[0] for k in st1.index]).sum()
print("\nrevenue unchanged after salting?", np.allclose(final.sort_index(), df.groupby("city").fare.sum().sort_index()))
print("\nSalting costs a second aggregation stage. It is worth it only when one key really does dominate.")